# Lab Tasks - Solutions

This task involves working with a dataset which contains information about contacts between patients and health-care workers in a hospital ward in France. The data was gathered using wearable sensors which were able to detect face-to-face close-range proximity between participants wearing the sensors.

For more details see:

- http://www.sociopatterns.org/datasets/hospital-ward-dynamic-contact-network/
- Génois, M., & Barrat, A. Can co-location be used as a proxy for face-to-face contacts?. EPJ Data Science, 7(1), 11. (2018)

The raw data is provided as 2 tab-separated files:

1. *hospital-metadata.csv*: One line per participant in the hospital ward, where each participant has a unique numeric ID and a "status" (e.g., patient, doctor, etc.).
2. *hospital-contacts.csv*: Each line indicates a contact event between two participants, with the time of the event and the numeric IDs of the participants involved.

In [ ]:
import networkx as nx
import pandas as pd

### Task 1

Load the hospital metadata and contact data into two separate Pandas DataFrames.

In [ ]:
# read the metadata
df_metadata = pd.read_csv("hospital-metadata.csv", sep="\t").set_index("participant_id")
df_metadata.head(5)

In [ ]:
# read the contact events
df_contacts = pd.read_csv("hospital-contacts.csv", sep="\t")
df_contacts.head(5)

### Task 2

Create a weighted undirected network from the two DataFrames, such that:

- There is a node for every participant in the study. Each node should have an attribute indicating their "status".
- Edges between nodes have a "weight", indicating the number of times two participants (nodes) have been in contact.

What size is the resulting network?

In [ ]:
# create the undirected network and add the nodes from the metadata
g = nx.Graph()
for participant_id, row in df_metadata.iterrows():
    g.add_node(participant_id, status=row["status"])

In [ ]:
# count frequency of contact between each pair of participants
from collections import Counter
counts = Counter()
for i, row in df_contacts.iterrows():
    node1 = row["participant1"]
    node2 = row["participant2"]
    # represent the pair as a frozen set, so it's unique and hashable
    pair = frozenset([node1, node2])
    counts[pair] += 1

In [ ]:
# add weighted edges for each pair
for p in counts:
    pair = list(p)
    node1, node2 = pair[0], pair[1]
    g.add_edge(node1, node2, weight=counts[p])

In [ ]:
# check size of the network
print(f"Network has {g.number_of_nodes()} nodes and {g.number_of_edges()} edges")

### Task 3

From the network created in Task 2, filter out all isolated nodes (i.e., nodes with degree 0).

In [ ]:
# calculate the degrees by degree
degrees = dict(g.degree())
# figure out which nodes we want to remove
degree_removals = []
for node in degrees:
    if degrees[node] < 1:
        degree_removals.append(node)

In [ ]:
# actually remove the nodes
print(f"Removing: {degree_removals}")
g.remove_nodes_from(degree_removals)
print(f"Filtered network has {g.number_of_nodes()} nodes, {g.number_of_edges()} edges")

### Task 4

Export the filtered network from Task 3 as a new GEXF file.

In [ ]:
# save the filtered network as a GEXF file
nx.write_gexf(g, "hospital.gexf")

### Task 5

Load the GEXF file from Task 4 in Gephi, and complete the following steps:
    
1. Colour the nodes in the network based on their "status" attribute, via the *Appearance Panel* on the *Overview* screen.
2. Calculate the **weighted degree** of the nodes via the *Statistics Panel* on the *Overview* screen.
3. Scale the size of the nodes via the *Appearance Panel* on the *Overview* screen, ranked based on their **weighted degree**.
4. Apply the **Force Atlas** algorithm to lay out the network on the *Overview* screen. Adjust the parameters on the *Layout* panel to improve the network's readability.
5. Filter the nodes in the network to only display nodes with the "status" attribute equal to "doctor", via the *Filters Panel* on the *Overview* screen.
6. Use the *Preview* screen to adjust the final appearance of the network, and then export an image of the network as a PNG file.

In [ ]:
## See hospital.gephi in solutions